# Response Generation Benchmark — Walkthrough
**Llama 3 8B vs Mistral 7B on EmpatheticDialogues / DailyDialog**

This notebook runs the full benchmark pipeline cell-by-cell, useful for
interactive exploration, debugging, and generating figures for the EMAH
thesis (Chapters 3–4).

**Pipeline:** dataset loading → prompt construction → Ollama inference →
metric scoring (BLEU, ROUGE-L, BERTScore, Empathy Score) → aggregation →
visualisation.

> Run this notebook from the `resp_gen_benchmark/` directory (or adjust
> `sys.path` in the first cell) so the `config`, `data_loader`, etc. modules
> are importable.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Locate the project root (the directory containing config.py) by searching
# upward from the current working directory. This works regardless of
# whether Jupyter's cwd is the repo root, notebooks/, or somewhere else.
def _find_project_root(start: Path, marker: str = "config.py", max_up: int = 5) -> Path:
    p = start.resolve()
    for _ in range(max_up + 1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(
        f"Could not locate '{marker}' within {max_up} levels above {start}. "
        f"Run this notebook from inside resp_gen_benchmark/ or its notebooks/ subfolder."
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from config import MODELS, MODEL_KEYS, DATASETS, GENERATION
from data_loader import load_dataset_samples
from model_client import get_client
from prompt_builder import build_full_prompt, SYSTEM_PROMPT
from evaluator import evaluate_batch, aggregate, load_empathy_classifier, SCORE_COLS
from visualize_results import (
    plot_bar_comparison, plot_radar, plot_latency_distribution,
    plot_scatter_bleu_rouge, plot_emotion_heatmap,
)

import matplotlib.pyplot as plt
%matplotlib inline

print("Models:", MODEL_KEYS)
print("Datasets:", list(DATASETS.keys()))


## 2. Configuration

Adjust these for your run. Defaults match `benchmark_runner.py`.


In [ ]:
DATASET_NAME   = "empathetic_dialogues"   # or "daily_dialog"
SPLIT          = "test"
NUM_SAMPLES    = 20                       # small for interactive runs; increase for full benchmark
MODE           = "zero_shot"              # "zero_shot" or "few_shot"
MAX_TOKENS     = GENERATION["max_tokens"]
SKIP_EMPATHY   = False
EMPATHY_DEVICE = "auto"                   # "auto" | "cpu" | "gpu"

OLLAMA_HOST = "http://localhost:11434"


## 3. Load Dataset

Downloads on first use (with retries and fallback sources) and caches
preprocessed samples under `data_cache/`. Re-running this cell with the
same parameters loads instantly from cache.


In [ ]:
samples = load_dataset_samples(
    name=DATASET_NAME,
    split=SPLIT,
    num_samples=NUM_SAMPLES,
)

print(f"Loaded {len(samples)} samples")
pd.DataFrame(samples).head()


### Emotion distribution in the sample

In [ ]:
emotion_counts = pd.Series([s["emotion"] for s in samples]).value_counts()
emotion_counts.plot(kind="bar", figsize=(10, 4), title="Emotion category distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## 4. Inspect a Sample Prompt

See exactly what's sent to the model for one sample.


In [ ]:
example = samples[0]
prompt = build_full_prompt(
    emotion=example["emotion"],
    context=example["context"],
    utterance=example["utterance"],
    mode=MODE,
)

print("=" * 60)
print("FULL PROMPT SENT TO MODEL")
print("=" * 60)
print(prompt)
print()
print("=" * 60)
print("GOLD REFERENCE RESPONSE")
print("=" * 60)
print(example["reference"])


## 5. Connect to Ollama

Health-checks the server and lists locally available models. Make sure
both `llama3:8b` and `mistral:7b` are pulled:

```bash
ollama pull llama3:8b
ollama pull mistral:7b
```


In [ ]:
client = get_client("ollama", host=OLLAMA_HOST)

if not client.health_check():
    raise RuntimeError(f"Cannot reach Ollama at {OLLAMA_HOST}. Is the server running?")

print("Ollama reachable ✓")
print("Available models:", client.list_models())


## 6. Run Inference

Generates a response for every sample with each model. This is the
slowest cell — progress bars show per-model status.


In [ ]:
generations = {}  # model_key -> list of (hypothesis, latency)

for model_key in MODEL_KEYS:
    label = MODELS[model_key]["label"]
    results = []
    for sample in tqdm(samples, desc=label):
        hyp, lat = client.generate(
            model_key=model_key,
            emotion=sample["emotion"],
            context=sample["context"],
            utterance=sample["utterance"],
            max_tokens=MAX_TOKENS,
        )
        results.append((hyp, lat))
    generations[model_key] = results

print("Done.")


## 7. Spot-Check Generations

In [ ]:
idx = 0  # change to inspect a different sample

print("EMOTION:   ", samples[idx]["emotion"])
print("UTTERANCE: ", samples[idx]["utterance"])
print("REFERENCE: ", samples[idx]["reference"])
print()
for model_key in MODEL_KEYS:
    hyp, lat = generations[model_key][idx]
    print(f"--- {MODELS[model_key]['label']} ({lat:.2f}s) ---")
    print(hyp)
    print()


## 8. Compute Metrics

BLEU, ROUGE-L, BERTScore F1, and Empathy Score (zero-shot NLI via
`facebook/bart-large-mnli`).


In [ ]:
empathy_clf = None
if not SKIP_EMPATHY:
    empathy_clf = load_empathy_classifier(EMPATHY_DEVICE)


In [ ]:
records = []
for model_key in MODEL_KEYS:
    hyps = [h for h, _ in generations[model_key]]
    lats = [l for _, l in generations[model_key]]
    refs = [s["reference"] for s in samples]

    metrics = evaluate_batch(refs, hyps, lats, empathy_clf)

    for i, sample in enumerate(samples):
        records.append({
            "model": model_key,
            "label": MODELS[model_key]["label"],
            "mode": MODE,
            "conv_id": sample["conv_id"],
            "emotion": sample["emotion"],
            "utterance": sample["utterance"],
            "reference": refs[i],
            "hypothesis": hyps[i],
            **{col: metrics[col][i] for col in SCORE_COLS},
        })

df = pd.DataFrame(records)
df.head()


## 9. Aggregate Results

In [ ]:
summary_rows = []
for model_key, grp in df.groupby("model"):
    row = {"model": model_key, "label": MODELS[model_key]["label"]}
    for col in SCORE_COLS:
        stats = aggregate(grp[col].tolist())
        row[f"{col}_mean"] = stats["mean"]
        row[f"{col}_std"]  = stats["std"]
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
emotion_rows = []
for (model_key, emotion), grp in df.groupby(["model", "emotion"]):
    row = {"model": model_key, "emotion": emotion, "n": len(grp)}
    for col in SCORE_COLS:
        row[f"{col}_mean"] = aggregate(grp[col].tolist())["mean"]
    emotion_rows.append(row)

emotion_df = pd.DataFrame(emotion_rows).sort_values(["emotion", "model"]).reset_index(drop=True)
emotion_df.head(10)


## 10. Visualise Results

The plotting functions from `visualize_results.py` save PNGs and close the
figure (matching CLI behaviour). We display the saved images inline below.


In [ ]:
from IPython.display import Image, display

fig_dir = PROJECT_ROOT / "_tmp_figures"
fig_dir.mkdir(exist_ok=True)

# Persist per-model raw CSVs so plot_latency_distribution /
# plot_scatter_bleu_rouge (which read from disk) can find them.
tmp_results_dir = PROJECT_ROOT / "_tmp_results"
tmp_results_dir.mkdir(exist_ok=True)
for model_key in MODEL_KEYS:
    df[df["model"] == model_key].to_csv(tmp_results_dir / f"raw_{model_key}.csv", index=False)


In [ ]:
plot_bar_comparison(summary, fig_dir)
display(Image(filename=fig_dir / "comparison_bar.png"))


In [ ]:
plot_radar(summary, fig_dir)
display(Image(filename=fig_dir / "radar_profile.png"))


In [ ]:
plot_latency_distribution(tmp_results_dir, fig_dir)
display(Image(filename=fig_dir / "latency_distribution.png"))


In [ ]:
plot_scatter_bleu_rouge(tmp_results_dir, fig_dir)
display(Image(filename=fig_dir / "scatter_bleu_rouge.png"))


In [ ]:
for metric in ["bleu", "rouge_l", "empathy_score"]:
    plot_emotion_heatmap(emotion_df, metric, fig_dir)
    img_path = fig_dir / f"emotion_heatmap_{metric}.png"
    if img_path.exists():
        display(Image(filename=img_path))


## 11. Save Results

Writes CSVs/JSON to `results/<dataset>/` and figures to `figures/<dataset>/`,
matching the layout produced by `benchmark_runner.py`.


In [ ]:
import json
import shutil

results_dir = PROJECT_ROOT / "results" / DATASET_NAME
figures_dir = PROJECT_ROOT / "figures" / DATASET_NAME
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(results_dir / "summary.csv", index=False)
emotion_df.to_csv(results_dir / "per_emotion_summary.csv", index=False)
for model_key in MODEL_KEYS:
    df[df["model"] == model_key].to_csv(results_dir / f"raw_{model_key}.csv", index=False)

def _nan_safe(v):
    return None if isinstance(v, float) and np.isnan(v) else v

with open(results_dir / "all_results.json", "w") as f:
    json.dump([{k: _nan_safe(v) for k, v in r.items()} for r in records], f, indent=2)

# Re-generate the full figure set into the canonical figures directory
from visualize_results import generate_all
generate_all(results_dir, figures_dir)

# Clean up notebook scratch directories
shutil.rmtree(tmp_results_dir, ignore_errors=True)
shutil.rmtree(fig_dir, ignore_errors=True)

print(f"Results saved to {results_dir.resolve()}/")
print(f"Figures saved to {figures_dir.resolve()}/")


## 12. Summary — Winner per Metric

In [ ]:
METRIC_LABELS = {
    "bleu_mean":          "BLEU ↑",
    "rouge_l_mean":       "ROUGE-L ↑",
    "bert_score_f1_mean": "BERTScore F1 ↑",
    "empathy_score_mean": "Empathy Score ↑",
    "latency_s_mean":     "Latency (s) ↓",
}

for col, label in METRIC_LABELS.items():
    if col not in summary.columns or summary[col].isna().all():
        continue
    if "latency" in col:
        winner = summary.loc[summary[col].idxmin()]
    else:
        winner = summary.loc[summary[col].idxmax()]
    print(f"{label:<22}: {winner['label'].upper():<12} ({winner[col]:.4f})")


## Next Steps

- Increase `NUM_SAMPLES` and re-run for the full benchmark (the CLI
  equivalent is `python benchmark_runner.py`).
- Set `MODE = "few_shot"` to compare prompting strategies.
- Switch `DATASET_NAME = "daily_dialog"` to benchmark on DailyDialog instead.
- For larger runs, prefer the CLI (`benchmark_runner.py --mode both
  --num_samples 200`) over this notebook — it's faster and saves
  intermediate state.
